# CREMA-D Data Preparation: Train/Test Split

This notebook splits the 7,442 WAV files from `AudioWAV/` into actor-disjoint train/test sets, organized by emotion subdirectory.

**Prerequisites:** Run `crema_d_setup.ipynb` first to clone the repo and pull LFS media files.

**Output:** `data/train/{emotion}/` and `data/test/{emotion}/` directories compatible with PyTorch `ImageFolder` / Keras `image_dataset_from_directory`.

Run all cells top-to-bottom from the **repo root directory**.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import shutil

REPO_ROOT = Path.cwd().resolve()
AUDIO_SRC = REPO_ROOT / "AudioWAV"
OUTPUT_DIR = REPO_ROOT / "data"
TEST_RATIO = 0.2
RANDOM_SEED = 42
EMOTIONS = ["ANG", "DIS", "FEA", "HAP", "NEU", "SAD"]

# Verify we're in the right place
assert AUDIO_SRC.exists(), f"AudioWAV/ not found at {AUDIO_SRC}. Run this notebook from the repo root."
print(f"Repo root: {REPO_ROOT}")
print(f"Audio source: {AUDIO_SRC}")
print(f"Output dir: {OUTPUT_DIR}")

Repo root: /Users/velu/Documents/github/crema-d-mirror
Audio source: /Users/velu/Documents/github/crema-d-mirror/AudioWAV
Output dir: /Users/velu/Documents/github/crema-d-mirror/data


## Load clip list

Read `SentenceFilenames.csv` and parse each base filename into its four components.
Filenames in the CSV have no extension — we append `.wav` when constructing source paths.

In [2]:
sf = pd.read_csv(REPO_ROOT / "SentenceFilenames.csv")
print(f"Clips in master list: {len(sf)}")

# Parse filename components
parts = sf["Filename"].str.split("_", expand=True)
sf["ActorID"] = parts[0].astype(int)
sf["SentenceCode"] = parts[1]
sf["EmotionCode"] = parts[2]
sf["LevelCode"] = parts[3]

# Verify source WAV exists for every clip
sf["wav_path"] = sf["Filename"].apply(lambda f: AUDIO_SRC / f"{f}.wav")
missing = sf[~sf["wav_path"].apply(lambda p: p.exists())]
if len(missing) > 0:
    print(f"WARNING: {len(missing)} WAV files missing!")
    print(missing["Filename"].head(10).tolist())
else:
    print("All 7,442 WAV files found.")

sf.head()

Clips in master list: 7442
All 7,442 WAV files found.


,Stimulus_Number,Filename,ActorID,SentenceCode,EmotionCode,LevelCode,wav_path
0,1,1001_IEO_NEU_XX,1001,IEO,NEU,XX,/Users/velu/Documents/github/crema-d-mirror/Au...
1,2,1001_IEO_HAP_LO,1001,IEO,HAP,LO,/Users/velu/Documents/github/crema-d-mirror/Au...
2,3,1001_IEO_HAP_MD,1001,IEO,HAP,MD,/Users/velu/Documents/github/crema-d-mirror/Au...
3,4,1001_IEO_HAP_HI,1001,IEO,HAP,HI,/Users/velu/Documents/github/crema-d-mirror/Au...
4,5,1001_IEO_SAD_LO,1001,IEO,SAD,LO,/Users/velu/Documents/github/crema-d-mirror/Au...


## Actor-disjoint split

Split the 91 actors into train (72) and test (19) groups using a fixed random seed.
The 80/20 ratio applies to actors, not clips — the actual clip-level ratio will be approximate.

In [3]:
demo = pd.read_csv(REPO_ROOT / "VideoDemographics.csv")
actor_ids = demo["ActorID"].values
assert len(actor_ids) == 91, f"Expected 91 actors, got {len(actor_ids)}"
print(f"Total actors: {len(actor_ids)}")

# Shuffle with fixed seed and split
rng = np.random.default_rng(RANDOM_SEED)
shuffled = actor_ids.copy()
rng.shuffle(shuffled)

n_train_actors = int(np.floor(len(shuffled) * (1 - TEST_RATIO)))
train_actors = set(shuffled[:n_train_actors])
test_actors = set(shuffled[n_train_actors:])

print(f"Train actors: {len(train_actors)}, Test actors: {len(test_actors)}")
assert len(train_actors & test_actors) == 0, "Actor overlap detected!"

# Assign split to each clip
sf["split"] = sf["ActorID"].apply(lambda a: "train" if a in train_actors else "test")

train_count = (sf["split"] == "train").sum()
test_count = (sf["split"] == "test").sum()
print(f"Train clips: {train_count} ({train_count/len(sf)*100:.1f}%)")
print(f"Test clips: {test_count} ({test_count/len(sf)*100:.1f}%)")
print(f"Total: {train_count + test_count}")

Total actors: 91
Train actors: 72, Test actors: 19
Train clips: 5890 (79.1%)
Test clips: 1552 (20.9%)
Total: 7442


## Copy WAV files into emotion subdirectories

Creates `data/{train,test}/{emotion}/` and copies each WAV to its target location.
This duplicates ~580 MB of audio. Uses `shutil.copy2` to preserve file metadata.

In [4]:
# Create directories
for split in ["train", "test"]:
    for emo in EMOTIONS:
        (OUTPUT_DIR / split / emo).mkdir(parents=True, exist_ok=True)

# Copy files
copied = 0
for _, row in sf.iterrows():
    src = row["wav_path"]
    dst = OUTPUT_DIR / row["split"] / row["EmotionCode"] / f"{row['Filename']}.wav"
    shutil.copy2(src, dst)
    copied += 1
    if copied % 1000 == 0:
        print(f"Copied {copied}/{len(sf)}...")

print(f"Done. Copied {copied} files.")

Copied 1000/7442...
Copied 2000/7442...
Copied 3000/7442...
Copied 4000/7442...
Copied 5000/7442...
Copied 6000/7442...
Copied 7000/7442...
Done. Copied 7442 files.


## Sanity checks

Verify the split is correct: total count, no actor overlap, emotion distribution.

In [5]:
# Count files on disk
train_files = list(OUTPUT_DIR.glob("train/*/*.wav"))
test_files = list(OUTPUT_DIR.glob("test/*/*.wav"))
total = len(train_files) + len(test_files)
print(f"Train files on disk: {len(train_files)}")
print(f"Test files on disk: {len(test_files)}")
print(f"Total: {total}")
assert total == 7442, f"Expected 7442, got {total}"

# Verify no actor overlap
train_actors_check = set(p.stem.split("_")[0] for p in train_files)
test_actors_check = set(p.stem.split("_")[0] for p in test_files)
overlap = train_actors_check & test_actors_check
assert len(overlap) == 0, f"Actor overlap: {overlap}"
print(f"Actor overlap check: PASS (0 shared actors)")

# Verify every file in SentenceFilenames.csv was placed
expected_stems = set(sf["Filename"])
actual_stems = set(p.stem for p in train_files + test_files)
missing_stems = expected_stems - actual_stems
extra_stems = actual_stems - expected_stems
assert len(missing_stems) == 0, f"Missing files: {missing_stems}"
assert len(extra_stems) == 0, f"Extra files: {extra_stems}"
print(f"Per-file placement check: PASS (all {len(expected_stems)} files accounted for)")

# Emotion distribution
print("\nEmotion distribution:")
print(f"{'Emotion':<10} {'Train':>8} {'Tr%':>6} {'Test':>8} {'Te%':>6} {'Total':>8}")
print("-" * 50)
for emo in EMOTIONS:
    tr = len(list((OUTPUT_DIR / "train" / emo).glob("*.wav")))
    te = len(list((OUTPUT_DIR / "test" / emo).glob("*.wav")))
    tr_pct = tr / len(train_files) * 100
    te_pct = te / len(test_files) * 100
    print(f"{emo:<10} {tr:>8} {tr_pct:>5.1f}% {te:>8} {te_pct:>5.1f}% {tr+te:>8}")

Train files on disk: 5890
Test files on disk: 1552
Total: 7442
Actor overlap check: PASS (0 shared actors)
Per-file placement check: PASS (all 7442 files accounted for)

Emotion distribution:
Emotion       Train    Tr%     Test    Te%    Total
--------------------------------------------------
ANG            1006  17.1%      265  17.1%     1271
DIS            1006  17.1%      265  17.1%     1271
FEA            1006  17.1%      265  17.1%     1271
HAP            1006  17.1%      265  17.1%     1271
NEU             860  14.6%      227  14.6%     1087
SAD            1006  17.1%      265  17.1%     1271
